# Peripheral immune response to severe COVID-19 — re-analysis of Wilk et al. 2020

**Paper:** Wilk AJ, Rustagi A, Zhao NQ, et al. "A single-cell atlas of the peripheral immune response in patients with severe COVID-19." *Nature Medicine* 26, 1070–1076 (2020). PMID 32514174.

**Data:** GEO `GSE150728` (raw) / COVID-19 Cell Atlas & CZ CELLxGENE (processed — use this one). Seq-Well scRNA-seq of PBMCs from 7 hospitalized COVID-19 patients (one, "C1", sampled at two timepoints — 9 and 11 days post-symptom-onset, before and after intubation) and 6 healthy controls. All 7 patients were male, aged 20 to 80+. Four of the eight COVID-19 samples came from ventilated ARDS patients; the other four were less severely ill.

**Question:** does peripheral immune composition and cell-cell signaling shift with severity, and does a Python/scanpy re-analysis reproduce the paper's central, somewhat counterintuitive claim — HLA class II *downregulation* and *low* (not elevated) pro-inflammatory cytokine expression in monocytes from severe patients?

The original analysis was done in R/Seurat (`github.com/ajwilk/2020_Wilk_COVID`). This is an independent re-analysis, not a line-by-line port — differences from the published numbers are expected and worth investigating, not evidence that either version is wrong.

**Setup** — packages this notebook uses:
```
pip install scanpy anndata pandas numpy matplotlib scrublet harmonypy celltypist milopy pydeseq2 liana-py leidenalg igraph
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

## Step 1 — Load the data

Two sources exist; the processed one is far less work:

- **Processed (recommended):** download the `.h5ad` for this collection from [CZ CELLxGENE](https://cellxgene.cziscience.com/d/Single_cell_atlas_of_peripheral_immune_response_to_SARS_CoV_2_infection-25.cxg/) or the [COVID-19 Cell Atlas](https://www.covid19cellatlas.org/#wilk20) — counts, per-cell metadata, and embeddings arrive already assembled.
- **Raw (more work, more control):** GEO `GSE150728` — 13 GSM samples (`GSM4557327`–`GSM4557339`), per-sample count matrices with no curated metadata; you'd rebuild the sample → patient → severity mapping by hand from the paper's Table 1, the way the GSE207633 practicum did with its clinical spreadsheet.

Set `DATA_PATH` below once downloaded. Everything past this cell assumes the processed `.h5ad`.

In [ ]:
DATA_PATH = "wilk_covid19_pbmc.h5ad"  # set to your downloaded file

adata = sc.read_h5ad(DATA_PATH)
print(adata)
print("\nobs columns:", list(adata.obs.columns))
adata.obs.head()

## Step 2 — Standardize sample / condition / severity labels

CZ CELLxGENE-curated objects carry standardized columns (typically `disease`, `donor_id`, `assay`) alongside whatever the submitters included. **Check the printout above before running the next cell** and edit `sample_col`/`condition_col` to match what's actually there.

Ventilation status (the paper's more informative severity axis — collapsing everything into "COVID vs. healthy" hides that non-ventilated and ventilated/ARDS patients look quite different) isn't a standardized field, so it needs the same by-hand treatment the GSE207633 practicum used for its clinical spreadsheet.

In [ ]:
sample_col = "donor_id"      # <- edit to match your actual obs.columns
condition_col = "disease"    # <- edit to match your actual obs.columns

adata.obs["sample"] = adata.obs[sample_col].astype(str)
adata.obs["condition"] = adata.obs[condition_col].astype(str).str.lower().map(
    lambda x: "healthy" if ("normal" in x or "healthy" in x) else "covid19"
)

# Fill in from adata.obs['sample'].unique() + the paper's Table 1 (ventilated/ARDS vs. not).
# Healthy samples are set automatically below — only the COVID rows need mapping.
ventilation_map = {}  # e.g. {"C1_d9": "non_ventilated", "C1_d11": "ventilated", "C2": "ventilated", ...}

adata.obs["severity"] = adata.obs["sample"].map(ventilation_map)
adata.obs.loc[adata.obs["condition"] == "healthy", "severity"] = "healthy"
adata.obs["severity"] = adata.obs["severity"].fillna("unknown")

adata.obs[["sample", "condition", "severity"]].drop_duplicates()

## Step 3 — QC metrics

One thing not to copy blindly from a 10x-tuned project: **Seq-Well has meaningfully lower per-cell UMI depth than droplet platforms.** A `total_counts > 500` cutoff that's reasonable for 10x data may be too aggressive here. Look at the plots below per sample before choosing thresholds.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P|E)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], percent_top=None, log1p=False, inplace=True)

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             groupby="sample", rotation=90, multi_panel=True)
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

## Step 4 — Filtering

The values below are a *starting point* informed by the Seq-Well depth caveat above, not a rule — raise or lower them based on where the distributions actually sit once you've looked at the plots.

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs["total_counts"] > 300, :]      # lower than the usual 10x-derived 500
adata = adata[adata.obs["total_counts"] < 25000, :]
adata = adata[adata.obs["n_genes_by_counts"] > 250, :]
adata = adata[adata.obs["n_genes_by_counts"] < 4000, :]
adata = adata[adata.obs["pct_counts_mt"] < 20, :]

print(adata.shape)

## Step 5 — Doublet detection

Run per `sample`, never on the merged object — pooling first biases the doublet model toward whichever sample has the most cells.

In [ ]:
sc.external.pp.scrublet(adata, batch_key="sample", threshold=0.25)
sc.external.pl.scrublet_score_distribution(adata)
print(adata.obs["predicted_doublet"].value_counts())

adata = adata[adata.obs["predicted_doublet"] == False].copy()

## Step 6 — Normalize, select HVGs, checkpoint

Log-normalization is a legitimate default here, not a compromise — Lause et al. (2021) found the shifted logarithm performs among the best of the preprocessing methods they tested. `batch_key` in the HVG call avoids selecting genes that look "variable" only because they differ between samples rather than between real cell states.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key="sample")
adata.write("wilk_covid_qc_normalized.h5ad")

## Step 7 — PCA on the HVG subset

Clustering and integration only need the ~3,000 HVGs, so this happens on a separate scaled copy — the full, unscaled, log-normalized `adata` stays untouched and comes back into play from Step 10 onward, since a gene excluded from the HVG set for low unsupervised variance can still be a real marker or a real ligand/receptor. Pick the PC count from the elbow in the variance-ratio plot (30–50 is typical), and check the first few PCs aren't just tracking `sample` or `pct_counts_mt` before trusting them.

In [ ]:
adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, svd_solver="arpack")

sc.pl.pca_variance_ratio(adata_hvg, n_pcs=40, log=True)
sc.pl.pca(adata_hvg, color=["sample", "pct_counts_mt"], ncols=2)

In [ ]:
N_PCS = 30  # set from the elbow plot above

sc.pp.neighbors(adata_hvg, n_neighbors=10, n_pcs=N_PCS)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg, resolution=0.5, key_added="leiden_preintegration")

sc.pl.umap(adata_hvg, color=["leiden_preintegration", "sample", "condition"], ncols=3, wspace=0.4)

## Step 8 — Batch integration

13 samples across two conditions is enough for `sample`-level batch effects to matter — check the UMAP above for whether cells are separating by `sample` rather than by biology before deciding integration is even necessary. Harmony is the fast default; the goal is an embedding that mixes well across `sample` while still separating cleanly by cell identity — good sample-mixing alone can also mean an overcorrected, biology-erasing embedding, so this needs a visual check both ways, not just one.

(Optional, worth doing before a real writeup: benchmark Harmony against `scvi.model.SCVI` with `scib.metrics.metrics_fast`, per `batch_integration.md` in the scrna-seq-workflow-guide skill — skippable for a first pass.)

In [ ]:
import scanpy.external as sce

sce.pp.harmony_integrate(adata_hvg, key="sample")
sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony", n_pcs=N_PCS)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg, resolution=0.5)   # final clustering, computed post-integration

sc.pl.umap(adata_hvg, color=["leiden", "sample", "condition"], ncols=3, wspace=0.4)

## Step 9 — Transfer graph-derived results back onto the full-gene object

Everything from here on (marker genes, differential expression, ligand-receptor scoring) needs the full transcriptome, not the 3,000-gene HVG subset used to build the graph.

In [ ]:
adata.obs["leiden"] = adata_hvg.obs["leiden"].values
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"]
adata.obsm["X_pca_harmony"] = adata_hvg.obsm["X_pca_harmony"]

adata.write("wilk_covid_clustered.h5ad")

## Step 10 — Marker genes + automatic annotation

Two independent signals, cross-checked against each other rather than either trusted alone: Wilcoxon marker genes per Leiden cluster, and CellTypist's pretrained immune model — a good fit here, since PBMC is exactly its training domain.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

canonical_markers = {
    "CD3D": "T cell", "CD3E": "T cell", "CD8A": "CD8 T", "CD4": "CD4 T", "IL7R": "CD4 T (memory)",
    "CD19": "B cell", "MS4A1": "B cell", "CD79A": "B cell",
    "CD14": "CD14 monocyte", "FCGR3A": "CD16 monocyte", "LYZ": "Monocyte / myeloid",
    "NKG7": "NK cell", "GNLY": "NK cell",
    "FCER1A": "Dendritic cell", "PPBP": "Platelet",
}
sc.pl.dotplot(adata, canonical_markers.keys(), groupby="leiden")

In [ ]:
import celltypist
from celltypist import models

models.download_models(model="Immune_All_Low.pkl")
model = models.Model.load("Immune_All_Low.pkl")

predictions = celltypist.annotate(adata, model=model, majority_voting=True)
adata.obs["celltypist_label"] = predictions.predicted_labels.majority_voting

pd.crosstab(adata.obs["leiden"], adata.obs["celltypist_label"])

Once the marker dotplot and the CellTypist crosstab agree on what each Leiden cluster is, assign readable labels. **Don't copy the mapping below verbatim** — fill it in from what you actually see; cluster numbering is arbitrary and will differ.

In [ ]:
cluster_labels = {
    "0": "CD14 monocyte", "1": "CD4 T cell", "2": "CD8 T cell", "3": "NK cell",
    "4": "B cell", "5": "CD16 monocyte", "6": "Dendritic cell", "7": "Platelet",
}  # placeholder — replace with your actual cluster -> label mapping

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_labels).fillna("unassigned")
sc.pl.umap(adata, color="cell_type")

## Step 11 — Composition: does cell-type proportion shift with severity?

Milo tests differential abundance on overlapping kNN-graph neighborhoods rather than discrete clusters — a better fit than a simple proportion test here, since healthy → non-ventilated → ventilated is closer to a severity gradient than three unrelated groups. This is also why filling in `severity` properly back in Step 2 matters — with everything left as `"unknown"`, this test has nothing to contrast.

In [ ]:
import milopy
import milopy.core as milo

sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_pcs=N_PCS)
milo.make_nhoods(adata)
milo.count_nhoods(adata, sample_col="sample")
milo.DA_nhoods(adata, design="~severity")

milo_results = adata.uns["nhood_adata"].obs
milo_results.sort_values("SpatialFDR").head(20)

## Step 12 — Pseudobulk differential expression: testing the paper's central claim

The paper's headline finding is that severe-COVID monocytes show **HLA class II *downregulation*** and **low, not elevated, pro-inflammatory cytokine expression** — a claim worth testing directly rather than taking on faith, and a genuinely useful one to disagree with if your numbers land differently. Cells from the same sample aren't independent replicates, so this has to be pseudobulk (sum raw counts per sample per cell type), not a per-cell Wilcoxon test — the single most common way scRNA-seq DE analyses overstate significance.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

mono = adata[adata.obs["cell_type"].str.contains("monocyte", case=False)]

counts = pd.DataFrame(
    mono.layers["counts"].toarray() if hasattr(mono.layers["counts"], "toarray") else mono.layers["counts"],
    index=mono.obs_names, columns=mono.var_names,
)
counts["sample"] = mono.obs["sample"].values
pb_counts = counts.groupby("sample").sum()

pb_meta = mono.obs[["sample", "condition"]].drop_duplicates().set_index("sample").loc[pb_counts.index]

# some pydeseq2 versions use metadata= instead of clinical= for this argument
dds = DeseqDataSet(counts=pb_counts.astype(int), clinical=pb_meta, design="~condition")
dds.deseq2()
stat_res = DeseqStats(dds, contrast=["condition", "covid19", "healthy"])
stat_res.summary()

hla_genes = [g for g in ["HLA-DRA", "HLA-DRB1", "HLA-DQA1", "HLA-DPA1"] if g in stat_res.results_df.index]
cytokine_genes = [g for g in ["IL1B", "IL6", "TNF", "CXCL8"] if g in stat_res.results_df.index]
stat_res.results_df.loc[hla_genes + cytokine_genes]

## Step 13 — Cell-cell communication: does inflammatory signaling change with condition?

LIANA scores ligand-receptor pairs per sender/receiver cell-type pair using several methods aggregated into one consensus rank. Running it separately on the healthy and COVID-19 subsets and comparing shows which communication programs appear or disappear with infection — a more direct read on "does the immune system talk to itself differently" than composition alone.

In [ ]:
import liana as li

results = {}
for cond in adata.obs["condition"].unique():
    sub = adata[adata.obs["condition"] == cond].copy()
    li.mt.rank_aggregate(sub, groupby="cell_type",
                          expr_props={"ligand": 0.1, "receptor": 0.1},
                          resource_name="consensus", verbose=False)
    results[cond] = sub.uns["liana_res"]

results["covid19"].sort_values("magnitude_rank").head(20)

In [ ]:
covid_pairs = set(zip(results["covid19"]["ligand_complex"], results["covid19"]["receptor_complex"],
                       results["covid19"]["source"], results["covid19"]["target"]))
healthy_pairs = set(zip(results["healthy"]["ligand_complex"], results["healthy"]["receptor_complex"],
                         results["healthy"]["source"], results["healthy"]["target"]))

covid_specific = covid_pairs - healthy_pairs
print(f"{len(covid_specific)} sender-receiver-ligand-receptor combinations detected only in the COVID-19 samples")
list(covid_specific)[:20]

## Where this leaves you

Checkpoints worth comparing against the paper directly, roughly in order of how distinctive the finding is:

1. The pseudobulk HLA-II and cytokine genes in Step 12 vs. the paper's claim of *low* (not elevated) monocyte cytokine expression — the result most worth reconciling if it disagrees, since it's the paper's most counterintuitive claim.
2. The Milo composition result in Step 11 vs. their reported shifts in peripheral immune composition.
3. The COVID-specific ligand-receptor pairs in Step 13 vs. their reported interferon-stimulated gene signature.

Natural next extensions, roughly in increasing effort: Tensor-cell2cell to factorize communication across all 13 samples by severity rather than two pooled groups (`advanced_analysis.md` §6.2 in the scrna-seq-workflow-guide skill), Augur to rank which cell type shows the strongest transcriptional response to infection, and a second integration method (scVI) benchmarked against Harmony with `scib` before leaning on either fully.